# A coherent wireless process

Two users harvest energy and transmit packets that expire after one slot. The analytical model and simulator share the outcome function; we explicitly separate the original ambiguous channel interpretations. No learning algorithm receives the transition table.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/decision_lab").is_dir()
sys.path.insert(0, str(ROOT / "src"))


def read(path):
    return json.loads((ROOT / path).read_text())

## Source accounting

In [2]:
from decision_lab.wireless import WirelessEnv, WirelessModel

model = WirelessModel()
state = model.index[(1, 2, 2, 1, 2, 2)]
served, expired, reward, valid = model.service(state, 3)
assert np.array_equal(served, [1, 1]) and reward == 0
served, expired, reward, valid = model.service(state, 0)
assert np.array_equal(expired, [1, 1]) and reward == -2
print("states", len(model.states), "legal state/action pairs", model.legal.sum())

states 324 legal state/action pairs 472


## Sampled conservation

In [3]:
env = WirelessEnv(model)
obs, info = env.reset(seed=23)
rng = np.random.default_rng(23)
for _ in range(1000):
    action = int(rng.choice(np.flatnonzero(info["action_mask"])))
    obs, reward, term, trunc, info = env.step(action)
    assert info["offered"] == info["served"] + info["expired"]
    assert reward == -info["expired"]
    if trunc:
        obs, info = env.reset()
env.close()
print("1000 conservation checks passed")

1000 conservation checks passed


## Two declared interpretations

In [4]:
planning = read("results/reference/planning/metrics.json")
pd.DataFrame(planning["rows"])[
    ["profile", "states", "normalized_optimal_loss", "average_packet_loss"]
]

,profile,states,normalized_optimal_loss,average_packet_loss
0,per_user,324,1.211381,1.224303
1,global,108,1.204254,1.218691


Full fits: `decision-lab planning --seed 0` (repeat seeds 1 and 2 where applicable). See [protocol](../docs/protocol.md), [results](../docs/results.md) and [references](../docs/references.md). These cells are numerical checks and evidence readers, not replacement training budgets.